# Check Recent Video Processing Status

This notebook checks if newly uploaded Keith Foskey videos were successfully processed and displays error logs if any failures occurred.

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import pandas as pd

# Load environment variables
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in environment variables")

# Create database connection
engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

print("✓ Database connection established")

✓ Database connection established


## Check Recent Videos from Playlist

Query the most recent videos ordered by publish date.

In [2]:
def check_recent_videos(days_back=7, limit=20):
    """Check videos from the last N days."""
    with SessionLocal() as session:
        # Get recent videos
        query = text("""
            SELECT 
                youtube_id,
                title,
                published_at,
                status,
                processed_at,
                error,
                created_at
            FROM videos
            ORDER BY published_at DESC NULLS LAST, created_at DESC
            LIMIT :limit
        """)
        
        result = session.execute(query, {"limit": limit})
        rows = result.fetchall()
        
        if not rows:
            print("No videos found in database")
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(rows, columns=[
            'youtube_id', 'title', 'published_at', 'status', 
            'processed_at', 'error', 'created_at'
        ])
        
        return df

# Get recent videos
recent_videos = check_recent_videos(days_back=30, limit=20)

if recent_videos is not None:
    print(f"Found {len(recent_videos)} videos\n")
    
    # Display summary by status
    print("Status Summary:")
    print(recent_videos['status'].value_counts())
    print("\n" + "="*80 + "\n")
    
    # Show all videos
    display(recent_videos[['youtube_id', 'title', 'published_at', 'status', 'processed_at']])

Found 20 videos

Status Summary:
status
processed    20
Name: count, dtype: int64




,youtube_id,title,published_at,status,processed_at
0,qtdwQvLhHmk,YourCalvinist Live with Keith & Jennifer Foskey,2026-01-28 04:36:28+00:00,processed,2026-01-30 18:44:19.624576+00:00
1,qS8kiCWbZy0,YourCalvinist LIVE with Keith & Jennifer Foskey,2026-01-21 04:41:18+00:00,processed,2026-01-24 20:49:22.525565+00:00
2,F5dkh7xt-4A,YourCalvinist Podcast LIVE Q&A with Keith & Je...,2026-01-14 04:17:15+00:00,processed,2026-01-17 20:55:19.482509+00:00
3,sKlD7dukxdQ,YourCalvinist LIVE with Keith & Jen Foskey,2026-01-07 04:26:33+00:00,processed,2026-01-17 20:57:47.648739+00:00
4,oGIqIuoBItQ,Last Live of '25! with Keith & Jen Foskey @You...,2025-12-31 04:34:33+00:00,processed,2026-01-17 21:00:07.134887+00:00
5,StUOq5SJed8,Live with Keith & Jen Foskey (@YourCalvinist),2025-12-17 04:31:44+00:00,processed,2026-01-17 21:03:08.442067+00:00
6,vEzTJ0s-oj8,Friday Night Live with Keith & Jen Foskey (@Yo...,2025-12-13 04:23:06+00:00,processed,2026-01-17 21:05:05.992289+00:00
7,xf0oztZTyzA,Live with Keith & Jen Foskey (@YourCalvinist P...,2025-12-10 04:13:10+00:00,processed,2026-01-17 21:06:56.714695+00:00
8,ZgY6K6B0YY4,Live with Keith & Jen Foskey (@YourCalvinist P...,2025-12-03 04:24:23+00:00,processed,2026-01-17 21:09:04.755415+00:00
9,yMCgAtKD5Ik,LIVE Q&A with Keith & Jennifer Foskey,2025-11-19 04:39:09+00:00,processed,2026-01-17 21:12:00.921548+00:00


## Check Missing Videos from Playlist

In [10]:
import sys
sys.path.insert(0, '/Users/spaceairmac/Dev/keith_foskey')

from app.youtube.playlist import get_playlist_video_ids
from app.settings import get_settings

settings = get_settings()

print("Fetching playlist videos...")
playlist_ids = get_playlist_video_ids()
playlist_len = len(playlist_ids) - 1

print(f"Playlist contains: {playlist_len} videos")

# Get database video IDs
with SessionLocal() as session:
    db_query = text("SELECT youtube_id FROM videos")
    db_videos = session.execute(db_query).fetchall()
    db_ids = set([row[0] for row in db_videos])

print(f"Database contains: {len(db_ids)} videos")
print(f"\nMissing from database: {playlist_len - len(db_ids)} videos\n")

# Find missing ones
missing = [vid for vid in playlist_ids if vid not in db_ids]

if missing:
    print("Missing video IDs:")
    for vid in missing:
        if vid != "4QpzXOyWDrE": # Confirmed with Keith to ignore this one since it's a duplicate
            print(f"  - {vid} (https://youtube.com/watch?v={vid})")
else:
    print("✓ All playlist videos are in the database!")

Fetching playlist videos...
Playlist contains: 74 videos
Database contains: 74 videos

Missing from database: 0 videos

Missing video IDs:


## 🔧 Manually Process a Single Video

Run a specific video through the ingestion pipeline directly.

In [9]:
# Process a specific video manually
from app.ingest.pipeline import process_video

# The video ID to process
VIDEO_ID = "pRVSnDoj8tE"

# First, let's check and clear any stale lock on this job
with SessionLocal() as session:
    from sqlalchemy import text
    
    # Check current job status
    check_query = text("""
        SELECT youtube_id, status, attempts, locked_at, last_error 
        FROM ingest_jobs 
        WHERE youtube_id = :vid
    """)
    result = session.execute(check_query, {"vid": VIDEO_ID}).fetchone()
    
    if result:
        print(f"Current job status:")
        print(f"  Status: {result.status}")
        print(f"  Attempts: {result.attempts}")
        print(f"  Locked at: {result.locked_at}")
        print(f"  Last error: {result.last_error}")
        
        # Clear the stale lock if it's stuck in 'processing'
        if result.status == 'processing':
            reset_query = text("""
                UPDATE ingest_jobs 
                SET status = 'pending', locked_at = NULL 
                WHERE youtube_id = :vid
            """)
            session.execute(reset_query, {"vid": VIDEO_ID})
            session.commit()
            print("\n✓ Reset stale lock - job is now pending")
    else:
        print("No ingest job found for this video ID")

print("\n" + "="*60)
print(f"Processing video: {VIDEO_ID}")
print("="*60 + "\n")

# Run the pipeline directly
result = process_video(VIDEO_ID, skip_classification=False, verbose=True)

print("\n" + "="*60)
if result.success:
    print(f"✓ Success! Saved {result.questions_saved} Q&A items")
else:
    print(f"✗ Failed: {result.error}")

Current job status:
  Status: failed
  Attempts: 3
  Locked at: None
  Last error: Failed to fetch transcript

Processing video: pRVSnDoj8tE

Processing: pRVSnDoj8tE
  Fetching metadata...
  Title: Your Calvinist Live with Keith & Jennifer Foskey
  Parsing timestamps...
  Found 13 timestamps
  Fetching transcript...
Transcript warning: SSL verify failed with proxy; retrying with SSL verification disabled


/Users/spaceairmac/.virtualenvs/foskey/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/spaceairmac/.virtualenvs/foskey/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/spaceairmac/.virtualenvs/foskey/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy-server.scraperapi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/late

  Transcript: 3412 segments
  Slicing answers...
  Classifying questions...
    [1/13] Who is your favorite puritan and why?...
    [2/13] Can you explain the last chapter of Levi...
    [3/13] When would you say a four-year-old is ab...
    [4/13] Can you define and explain the doctrine ...
    [5/13] How can you know the sheep from the wolv...
    [6/13] How would you respond to starting a dayc...
    [7/13] Is Elder Disqualification Permanent?...
    [8/13] Does Pornography Disqualify from Ministr...
    [9/13] Does Praying Change "Fate"?...
    [10/13] Is incrementalism biblical when addressi...
    [11/13] If a believer's sins are forgiven, why d...
    [12/13] What is your opinion about the Catholic ...
    [13/13] Question about an Application of the Fee...
  Saving to database...
  ✓ Saved 13 Q&A items

✓ Success! Saved 13 Q&A items
